# 초기 공통 기능 검수 — 김대건

이 노트북은 n=6, seed=0, 예산 20/40%의 **smoke**만 실행한다. 본 실험·규모 확보·최종 팀원 연구 제출이 아니다. 공통 생성법은 Dong·Tian (2022), DOI **10.1155/2022/3108327**을 참고한다. 구현 버전은 generation-v1 / evaluation-v2이며 실제 근거는 아래 저장 입력, 정확 탐색 RunRecord, 재채점된 참조와 CSV다. 전체 논문 방호 실험 재현, 실제 통신망/포탄 대표성 또는 최적 설정을 주장하지 않는다.

팀원 1/2/3: 실명·전략명·버전·논문 설명·실제 근거는 각 담당자가 `algorithms/member_1.py`~`member_3.py`의 설명란에 기입할 외부 의존성이다. 본문을 대신 구현하지 않으며 미구현은 성능 비교에서 제외한다.

In [1]:
# 담당: 김대건
# 과학 라이브러리보다 먼저 초기화한다. CLI subprocess도 각각 초기화한다.
from protection.runner import initialize_single_thread
initialize_single_thread()
from pathlib import Path
import json, subprocess, sys, tempfile, csv
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs/smoke.json').exists()
(ROOT / 'results/notebook-smoke').mkdir(parents=True, exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='run-', dir=ROOT / 'results/notebook-smoke'))
def cli(*args, expected=0):
    p = subprocess.run([sys.executable, '-m', 'protection.cli', *map(str, args)], cwd=ROOT, text=True, capture_output=True, timeout=180)
    assert p.returncode == expected, p.stdout + p.stderr
    print(p.stdout.strip())
    return p
print('Smoke artifacts:', WORK)

Smoke artifacts: /root/my-project/results/notebook-smoke/run-_8vep6jf


## A. 사전 준비 (공통 담당)
생성·저장 → 두 예산 단순 완전탐색 → 완료 참조 결합 → 재채점 검증. 중간 입력을 정식 참조 포함 배포본으로 표시하지 않는다.

In [2]:
# 담당: 김대건
prep = json.loads((ROOT / 'configs/smoke.json').read_text())
prep['output_dir'] = str(WORK / 'bank')
preparation = WORK / 'preparation.json'
preparation.write_text(json.dumps(prep, ensure_ascii=False))
cli('generate', '--manifest', preparation)
source = WORK / 'bank/inputs/n6-seed0.json'
assert source.exists()

{"artifact_kind":"scenario_input","inputs":["/root/my-project/results/notebook-smoke/run-_8vep6jf/bank/inputs/n6-seed0.json"]}


In [3]:
# 담당: 김대건
bank = []
exact_records = []
for percent in (20, 40):
    record = WORK / f'exact-{percent}.json'
    bundle = WORK / f'case-{percent}.json'
    cli('exact', '--input', source, '--budget-percent', percent, '--method', 'exhaustive', '--run-kind', 'smoke', '--limit-seconds', 120, '--output', record)
    cli('bundle', '--input', source, '--exact-record', record, '--output', bundle)
    checked = json.loads(cli('validate', '--bundle', bundle, '--require-reference').stdout)
    bank.append({'path': str(bundle), 'case_id': checked['case_id']})
    exact_records.append(json.loads(record.read_text()))
assert len({b['case_id'] for b in bank}) == 2
assert all(r['status'] == 'completed' and r['optimality_proven'] for r in exact_records)

{"output":"/root/my-project/results/notebook-smoke/run-_8vep6jf/exact-20.json","status":"completed"}


{"case_id":"f4cba3df80746adb545b1cd8897fdaf5ad4dc31c7e0846353eff91d3e9298704","output":"/root/my-project/results/notebook-smoke/run-_8vep6jf/case-20.json"}


{"case_id":"f4cba3df80746adb545b1cd8897fdaf5ad4dc31c7e0846353eff91d3e9298704","has_reference":true,"valid":true}


{"output":"/root/my-project/results/notebook-smoke/run-_8vep6jf/exact-40.json","status":"completed"}


{"case_id":"005e377cb7b23c1e700cf5ff540b7ad73f078e790d4f3e136d7c017994090a71","output":"/root/my-project/results/notebook-smoke/run-_8vep6jf/case-40.json"}


{"case_id":"005e377cb7b23c1e700cf5ff540b7ad73f078e790d4f3e136d7c017994090a71","has_reference":true,"valid":true}


## B. 저장 문제은행의 단일 파일 채점
이 구역은 생성기·정확 탐색을 호출하지 않는다. 실제 member_1 파일을 실행하여 NotImplementedError를 **not_implemented**로 기록한다. 다른 팀원은 미제출이며 대체 해나 성능을 만들지 않는다. 최적 참조는 채점 측만 사용하고 solve 입력에서 제외된다.

In [4]:
# 담당: 김대건
grading = json.loads((ROOT / 'configs/smoke-grading.json').read_text())
grading['bank'] = bank
grading['submission']['file'] = str(ROOT / 'algorithms/member_1.py')
grading['output_dir'] = str(WORK / 'member_1')
manifest = WORK / 'grading.json'
manifest.write_text(json.dumps(grading, ensure_ascii=False))
cli('grade', '--manifest', manifest, expected=1)
runs = [json.loads(line) for line in (WORK / 'member_1/runs.jsonl').read_text().splitlines()]
assert len(runs) == 2 and all(r['status'] == 'not_implemented' and r['submitted_evaluation'] is None for r in runs)
print('팀원 2/3: 미제출 / 전략·실명·연구 설명은 각 담당자가 작성; 비교 대상에서 제외')

{"output_dir":"/root/my-project/results/notebook-smoke/run-_8vep6jf/member_1","statuses":["not_implemented","not_implemented"]}
팀원 2/3: 미제출 / 전략·실명·연구 설명은 각 담당자가 작성; 비교 대상에서 제외


In [5]:
# 담당: 김대건
cli('report', '--records', WORK / 'member_1/runs.jsonl', '--bank-manifest', manifest, '--output-dir', WORK / 'report')
from protection.models import from_data, RunRecord, load_bundle
from protection.reporting import write_report
validated_bank = {b['case_id']: load_bundle(b['path'], require_reference=True) for b in bank}
write_report([from_data(RunRecord, r) for r in exact_records], validated_bank, WORK / 'exact-report')
def table(path, columns):
    rows = list(csv.DictReader(path.open()))
    text = '| ' + ' | '.join(columns) + ' |' + chr(10) + '| ' + ' | '.join('---' for _ in columns) + ' |' + chr(10)
    text += chr(10).join('| ' + ' | '.join(str(r[c]) for c in columns) + ' |' for r in rows)
    display(Markdown(text))
# 실제 파일의 값만 표시; 실패 점수는 빈 셀이다.
table(WORK / 'exact-report/runs.csv', ['case_id', 'status', 'connected_pairs', 'delivery_units', 'cost'])
table(WORK / 'report/runs.csv', ['case_id', 'status', 'connected_pairs', 'delivery_units'])
table(WORK / 'report/summary.csv', ['case_id', 'success_count', 'total_count', 'connected_ratio_median', 'delivery_ratio_median'])
print('NOTEBOOK_SMOKE_VERIFIED')

{"output_dir":"/root/my-project/results/notebook-smoke/run-_8vep6jf/report","record_count":2}


/root/my-project/protection/flow.py:191: OptimizeWarning: Unrecognized options detected: {'threads': 1}. These will be passed to HiGHS verbatim.
  result = linprog(lp.c, A_ub=lp.G, b_ub=lp.b,


| case_id | status | connected_pairs | delivery_units | cost |
| --- | --- | --- | --- | --- |
| f4cba3df80746adb545b1cd8897fdaf5ad4dc31c7e0846353eff91d3e9298704 | completed | 10 | 316456 | 10 |
| 005e377cb7b23c1e700cf5ff540b7ad73f078e790d4f3e136d7c017994090a71 | completed | 15 | 658228 | 22 |

| case_id | status | connected_pairs | delivery_units |
| --- | --- | --- | --- |
| f4cba3df80746adb545b1cd8897fdaf5ad4dc31c7e0846353eff91d3e9298704 | not_implemented |  |  |
| 005e377cb7b23c1e700cf5ff540b7ad73f078e790d4f3e136d7c017994090a71 | not_implemented |  |  |

| case_id | success_count | total_count | connected_ratio_median | delivery_ratio_median |
| --- | --- | --- | --- | --- |
| f4cba3df80746adb545b1cd8897fdaf5ad4dc31c7e0846353eff91d3e9298704 | 0 | 1 |  |  |
| 005e377cb7b23c1e700cf5ff540b7ad73f078e790d4f3e136d7c017994090a71 | 0 | 1 |  |  |

NOTEBOOK_SMOKE_VERIFIED
